In [1]:
!pip install yfinance feedparser transformers datasets torch accelerate scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 2.5 MB/s eta 0:00:00


In [2]:
#RSS FEED FETCHING(RSS -> REALLY SIMPLY SYNDICATE, TAKES XML FILE FOR REAL TIME FEED FOR FINANCIAL NEWS WRT OUR PROJECT!)
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'AMZN']

RSS_FEEDS = [
    "https://finance.yahoo.com/rssindex",
    "https://www.cnbc.com/id/100003114/device/rss/rss.html",   # CNBC markets
    "https://feeds.marketwatch.com/marketwatch/topstories/",
]

In [3]:
import yfinance as yf
import pandas as pd

def get_historical_data(tickers, period="180d"):
    data = yf.download(tickers, period=period, progress=False)['Close']
    data = data.dropna(axis=1, how='all')  # drop tickers that returned nothing (delisted etc.)
    missing = set(tickers) - set(data.columns)
    if missing:
        print(f"⚠️ Skipped (no data): {missing}")
    return data

price_data = get_historical_data(TICKERS)
returns = price_data.pct_change().dropna()

mu = returns.mean() * 252
Sigma = returns.cov() * 252
print(mu)

/tmp/ipykernel_738/1467081477.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, progress=False)['Close']


Ticker
AAPL     0.259755
AMZN     0.354544
GOOGL    0.368226
MSFT     0.091389
TSLA    -0.176576
dtype: float64


In [4]:
import feedparser
import asyncio
import aiohttp

async def fetch_feed(session, url):
    try:
        async with session.get(url, timeout=10) as resp:
            text = await resp.text()
            return feedparser.parse(text).entries
    except Exception as e:
        print(f"⚠️ Failed: {url} ({e})")
        return []

async def fetch_all_feeds(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_feed(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
    return [entry for feed in results for entry in feed]

all_entries = await fetch_all_feeds(RSS_FEEDS)
print(f"Pulled {len(all_entries)} headlines total")
for e in all_entries:
    print(e.get('title', ''))

Pulled 10 headlines total
My adult kids are big earners. Should I do a Roth conversion now so they aren’t taxed as much on their inheritance?
My brother asked me to sign as his executor, but wouldn’t let me read the document. Should I have refused?
‘This has been an emotionally difficult time’: My brother has cancer and my father is 94. How do I shoulder this responsibility?
The size of the American workforce has fallen by over 1 million people in the past year. Here’s what’s going on.
‘My head hit the driver’s window’: A teenager T-boned our car. Will Medicare pay my hospital bills?
‘My wife and I are both retired’: Do we dip into our $2.3 million fund to pay off our $300,000 mortgage at 2.9%?
Paying for pricey daycare? Here’s how to keep saving for retirement at the same time.
Should wealthier Americans forgo their Social Security benefits as a charitable gesture?
Changing how Medicare pays for hospice care could save the U.S. $7.6 billion a year — but at what cost to patients?
Berks

In [5]:
def ticker_rss_url(ticker):
    return f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US"

TICKER_FEEDS = {t: ticker_rss_url(t) for t in TICKERS}

async def fetch_ticker_news(session, ticker, url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        async with session.get(url, timeout=10, headers=headers) as resp:
            text = await resp.text()
            entries = feedparser.parse(text).entries
            titles = [e.get('title', '') for e in entries[:5]]
            print(f"✅ {ticker} → {len(titles)} headlines")
            return ticker, titles
    except Exception as e:
        print(f"⚠️ {ticker} failed: {e}")
        return ticker, []

async def fetch_all_ticker_news(ticker_feeds):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_ticker_news(session, t, url) for t, url in ticker_feeds.items()]
        results = await asyncio.gather(*tasks)
    return dict(results)

news_map = await fetch_all_ticker_news(TICKER_FEEDS)
news_map

✅ GOOGL → 5 headlines
✅ TSLA → 5 headlines
✅ MSFT → 5 headlines
✅ AMZN → 5 headlines
✅ AAPL → 5 headlines


{'AAPL': ['Bank of America revamps Apple stock price target after earnings',
  "Apple says Mac users in China can connect to Alibaba's Qwen AI service",
  'Apple (AAPL) Q3 2026 Earnings Call Transcript',
  'Apple and OpenAI escalate legal battle over devices',
  '‘Your refund: $0’: Companies claim billions in tariff refunds, months after Trump floated $2,000 checks'],
 'MSFT': ['AI is Hollowing Out Tech Sector Jobs: Oracle and Microsoft Help Push Layoff Rate to 20-Year High',
  'How Chevron became the AI darling of Big Oil',
  '7 Reasons Pershing Square (PS) Looks Pricey After Ackman Spotlighted AI Holdings',
  'How a 52-Year-Old Can Turn $425,000 Into a Monthly Paycheck Machine by 62',
  'Osisko Metals (TSX:OM) Is Up 20.8% After Reporting Deepening 2026 Losses Has The Bull Case Changed?'],
 'GOOGL': ['Berkshire Hathaway Just Did Something It Hasn’t Done in More Than 3 Years',
  'Alphabet CEO Sundar Pichai Just Made a Decision That Could Move Nvidia and Broadcom Stock',
  'Google shutt

In [6]:
#Loading the dataset
#Financial Phrasebook has all the data or financial newssentences
#sentences allagree means only sentences human annonators agreed to label
from datasets import load_dataset
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

dataset = load_dataset("atrost/financial_phrasebank")
dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
print(dataset)
print(dataset['train'][0])

README.md:   0%|          | 0.00/721 [00:00<?, ?B/s]

data/train-00000-of-00001-138b53eb17a3e8(…): reconstructing file:   0%|          |  0.00B /  268kB            

data/train-00000-of-00001-138b53eb17a3e8(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-0876be41e(…): reconstructing file:   0%|          |  0.00B / 68.7kB            

data/validation-00000-of-00001-0876be41e(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-41c7ea948573445(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/776 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2480
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 620
    })
})
{'sentence': "With Aldata , we can help retailers ensure that their merchandising and selections reflect customer demand and their evolving shopping behavior . ''", 'label': 1}


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch['sentence'], padding='max_length', truncation=True, max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2480 [00:00<?, ? examples/s]

Map:   0%|          | 0/620 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average='weighted')}

args = TrainingArguments(
    output_dir="./finbert-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.454624,0.414430,0.845161,0.842683
2,0.231276,0.502406,0.856452,0.856558
3,0.036456,0.561583,0.845161,0.847272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=465, training_loss=0.30826530488588477, metrics={'train_runtime': 219.5029, 'train_samples_per_second': 33.895, 'train_steps_per_second': 2.118, 'total_flos': 489390956974080.0, 'train_loss': 0.30826530488588477, 'epoch': 3.0})

In [9]:
from transformers import pipeline

# 🔧 Fix label mapping — dataset order (negative/neutral/positive)
# doesn't match FinBERT's original internal order
label_names = dataset['train'].features['label'].names
trainer.model.config.id2label = {i: l for i, l in enumerate(label_names)}
trainer.model.config.label2id = {l: i for i, l in enumerate(label_names)}

base_model = pipeline("sentiment-analysis", model="ProsusAI/finbert")
finetuned_model = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer)

test_headlines = [
    "Company shares plunge after missing earnings expectations",
    "Analysts raise price target following strong quarterly results",
    "Firm announces restructuring amid declining revenue"
]

for h in test_headlines:
    print(f"Headline: {h}")
    print("  Base FinBERT:      ", base_model(h))
    print("  Fine-tuned FinBERT:", finetuned_model(h))
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Headline: Company shares plunge after missing earnings expectations
  Base FinBERT:       [{'label': 'negative', 'score': 0.9621767401695251}]
  Fine-tuned FinBERT: [{'label': 'negative', 'score': 0.8890232443809509}]

Headline: Analysts raise price target following strong quarterly results
  Base FinBERT:       [{'label': 'positive', 'score': 0.914763867855072}]
  Fine-tuned FinBERT: [{'label': 'positive', 'score': 0.9147971272468567}]

Headline: Firm announces restructuring amid declining revenue
  Base FinBERT:       [{'label': 'negative', 'score': 0.9666624665260315}]
  Fine-tuned FinBERT: [{'label': 'negative', 'score': 0.8315339684486389}]



In [10]:
#for saving fine tuned model and tokenizer
trainer.save_model("/content/finbert-finetuned-final")
tokenizer.save_pretrained("/content/finbert-finetuned-final")

!zip -r finbert-finetuned-final.zip /content/finbert-finetuned-final
from google.colab import files
files.download("finbert-finetuned-final.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: content/finbert-finetuned-final/ (stored 0%)
  adding: content/finbert-finetuned-final/tokenizer_config.json (deflated 43%)
  adding: content/finbert-finetuned-final/training_args.bin (deflated 54%)
  adding: content/finbert-finetuned-final/config.json (deflated 54%)
  adding: content/finbert-finetuned-final/tokenizer.json (deflated 71%)
  adding: content/finbert-finetuned-final/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
eval_results = trainer.evaluate()
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.036456,0.414430,3,0.845161,0.842683


{'eval_loss': 0.41443008184432983, 'eval_accuracy': 0.8451612903225807, 'eval_f1': 0.8426830900804585}


In [12]:
#PHASE 2(SENTIMENT TO UB_i)

In [13]:
print('trainer' in dir())
print('mu' in dir())
print('news_map' in dir())

True
True
True


In [14]:
!pip install qiskit qiskit-aer --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.1 MB/s eta 0:00:00


In [15]:
import numpy as np

def get_sentiment_score(headlines, model):
    if not headlines:
        return 0.0  # no news = neutral, no adjustment
    scores = []
    for h in headlines:
        result = model(h)[0]
        label = result['label']
        conf = result['score']
        if label == 'positive':
            scores.append(conf)
        elif label == 'negative':
            scores.append(-conf)
        else:  # neutral
            scores.append(0.0)
    return float(np.mean(scores))

S = np.array([get_sentiment_score(news_map[t], finetuned_model) for t in TICKERS])
print(dict(zip(TICKERS, S.round(3))))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'AAPL': np.float64(0.092), 'MSFT': np.float64(-0.08), 'GOOGL': np.float64(-0.043), 'TSLA': np.float64(-0.191), 'AMZN': np.float64(0.159)}


In [16]:
BASE_LIMIT = 0.35   # normal max allocation per stock, before sentiment adjusts it

UB = BASE_LIMIT * (1 + S)
UB = np.clip(UB, 0.05, 1.0)   # safety floor/ceiling — never let a cap hit 0% or exceed 100%

print(dict(zip(TICKERS, UB.round(3))))

{'AAPL': np.float64(0.382), 'MSFT': np.float64(0.322), 'GOOGL': np.float64(0.335), 'TSLA': np.float64(0.283), 'AMZN': np.float64(0.406)}


In [17]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

sim = AerSimulator()

def quantum_levy_step(dim, scale=0.01):
    qc = QuantumCircuit(dim, dim)
    qc.h(range(dim))              # put every qubit into superposition
    qc.measure(range(dim), range(dim))
    result = sim.run(qc, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    bits = np.array([int(b) for b in bitstring], dtype=float)
    return (bits * 2 - 1) * scale   # map 0/1 → -1/+1, scale to a small step

In [18]:
print(quantum_levy_step(len(TICKERS)))

[-0.01 -0.01  0.01 -0.01  0.01]


In [19]:
RISK_FREE = 0.02

def sharpe_fitness(w, mu, Sigma):
    w = np.clip(w, 0, None)
    if w.sum() == 0:
        return -999
    w = w / w.sum()
    ret = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    return -999 if vol == 0 else (ret - RISK_FREE) / vol

In [20]:
def run_hho(mu, Sigma, UB, dim, pop_size=25, max_iter=40, use_quantum=True):
    X = np.random.rand(pop_size, dim)
    X = X / X.sum(axis=1, keepdims=True)

    best, best_fit = X[0].copy(), -999

    for t in range(max_iter):
        E0 = np.random.uniform(-1, 1)
        E = 2 * E0 * (1 - t / max_iter)
        mean_pos = X.mean(axis=0)

        for i in range(pop_size):
            q = np.random.rand()
            if abs(E) >= 1:                                       # Exploration
                if q < 0.5:
                    X[i] = np.random.rand(dim)                     # random spot
                else:
                    X[i] = mean_pos - np.random.rand() * abs(mean_pos - X[i])   # follow swarm
            else:                                                  # Exploitation
                r = np.random.rand()
                step = (quantum_levy_step(dim) if use_quantum else np.random.uniform(-0.01, 0.01, dim))
                if abs(E) >= 0.5 and r >= 0.5:                      # Soft besiege
                    X[i] = best - E * abs(best - X[i])
                elif abs(E) < 0.5 and r >= 0.5:                     # Hard besiege
                    X[i] = best - E * abs(best - mean_pos)
                elif abs(E) >= 0.5 and r < 0.5:                     # Soft + Levy
                    X[i] = best - E * abs(best - X[i]) + step
                else:                                               # Hard + Levy
                    X[i] = best - E * abs(best - mean_pos) + step

            X[i] = np.clip(X[i], 0, UB)                             # 🔒 sentiment guardrail
            if X[i].sum() > 0:
                X[i] = X[i] / X[i].sum()

            fit = sharpe_fitness(X[i], mu, Sigma)
            if fit > best_fit:
                best_fit, best = fit, X[i].copy()

    return best, best_fit

In [21]:
best_weights, best_sharpe = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=True)
print("Best Sharpe:", round(best_sharpe, 3))
print(dict(zip(TICKERS, best_weights.round(3))))

Best Sharpe: 1.383
{'AAPL': np.float64(0.422), 'MSFT': np.float64(0.262), 'GOOGL': np.float64(0.316), 'TSLA': np.float64(0.0), 'AMZN': np.float64(0.0)}


In [22]:
quantum_results = []
classical_results = []

for run in range(5):
    _, q_sharpe = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=True)
    _, c_sharpe = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=False)
    quantum_results.append(q_sharpe)
    classical_results.append(c_sharpe)
    print(f"Run {run+1}: Quantum={q_sharpe:.3f}  Classical={c_sharpe:.3f}")

print(f"\nQuantum   — mean: {np.mean(quantum_results):.3f}, std: {np.std(quantum_results):.3f}")
print(f"Classical — mean: {np.mean(classical_results):.3f}, std: {np.std(classical_results):.3f}")

Run 1: Quantum=1.386  Classical=1.379
Run 2: Quantum=1.379  Classical=1.373
Run 3: Quantum=1.389  Classical=1.379
Run 4: Quantum=1.383  Classical=1.377
Run 5: Quantum=1.390  Classical=1.387

Quantum   — mean: 1.385, std: 0.004
Classical — mean: 1.379, std: 0.004


In [23]:
quantum_results = []
classical_results = []

for run in range(15):
    _, q_sharpe = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=True)
    _, c_sharpe = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=False)
    quantum_results.append(q_sharpe)
    classical_results.append(c_sharpe)

print(f"Quantum   — mean: {np.mean(quantum_results):.3f}, std: {np.std(quantum_results):.3f}, min: {min(quantum_results):.3f}")
print(f"Classical — mean: {np.mean(classical_results):.3f}, std: {np.std(classical_results):.3f}, min: {min(classical_results):.3f}")

# how many classical runs fell notably below the typical ~1.38 range?
outliers = [c for c in classical_results if c < 1.3]
print(f"Classical runs below 1.3: {len(outliers)} out of 15")

Quantum   — mean: 1.383, std: 0.006, min: 1.375
Classical — mean: 1.384, std: 0.005, min: 1.376
Classical runs below 1.3: 0 out of 15


In [24]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(quantum_results, classical_results)
print(f"t-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")

t-statistic: -0.646, p-value: 0.5232


In [25]:
from scipy import stats

levene_stat, levene_p = stats.levene(quantum_results, classical_results)
print(f"Levene's statistic: {levene_stat:.3f}, p-value: {levene_p:.4f}")

Levene's statistic: 1.409, p-value: 0.2453


In [26]:
quantum_results, classical_results = [], []
for run in range(50):
    _, q = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=True)
    _, c = run_hho(mu.values, Sigma.values, UB, dim=len(TICKERS), use_quantum=False)
    quantum_results.append(q)
    classical_results.append(c)

print(stats.ttest_ind(quantum_results, classical_results))
print(stats.levene(quantum_results, classical_results))

TtestResult(statistic=np.float64(0.3751138415723849), pvalue=np.float64(0.7083864102092272), df=np.float64(98.0))
LeveneResult(statistic=np.float64(0.5237877560046517), pvalue=np.float64(0.4709544433410736))


In [27]:
!pip install anthropic --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.5 MB/s eta 0:00:00


In [28]:
!pip install transformers accelerate --quiet

from transformers import pipeline
import torch

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [32]:
briefing = generate_briefing(
    weights=best_weights,
    sharpe=best_sharpe,
    tickers=TICKERS,
    sentiment_scores=S,
    ub_array=UB,
    used_quantum=True
)
print(briefing)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In this portfolio report, we've optimized the asset allocation to maximize returns while minimizing risk, employing Harris Hawks Swarm Optimization—a sophisticated algorithm known for its efficiency in complex problem-solving. To ensure the strategy stays within a sentiment-based guardrail, we've integrated a unique approach leveraging the unpredictable yet deterministic nature of quantum mechanics through Qiskit, a powerful quantum computing framework. This combination allows us to navigate the volatile market landscape with precision and adaptability.

Sharpe Ratio: 1.383

- AAPL: 42.2% allocated | sentiment +0.09 | cap 38.2%
- MSFT: 26.2% allocated | sentiment -0.08 | cap 32.2%
- GOOGL: 31.6% allocated | sentiment -0.04 | cap 33.5%
- TSLA: 0.0% allocated | sentiment -0.19 | cap 28.3%
- AMZN: 0.0% allocated | sentiment +0.16 | cap 40.6%



In [34]:
def run_full_pipeline(tickers, use_quantum=True):
    print("1️⃣ Fetching price data...")
    price_data = get_historical_data(tickers)
    returns = price_data.pct_change().dropna()
    mu_local = returns.mean() * 252
    Sigma_local = returns.cov() * 252
    assert not mu_local.isna().any(), "❌ mu contains NaNs — check tickers"

    print("2️⃣ Scoring sentiment...")
    sentiment = np.array([get_sentiment_score(news_map.get(t, []), finetuned_model) for t in tickers])

    print("3️⃣ Computing guardrail...")
    ub_local = np.clip(BASE_LIMIT * (1 + sentiment), 0.05, 1.0)

    print("4️⃣ Running optimizer...")
    weights, sharpe = run_hho(mu_local.values, Sigma_local.values, ub_local, dim=len(tickers), use_quantum=use_quantum)
    assert abs(weights.sum() - 1.0) < 0.01, "❌ weights don't sum to ~1"
    assert sharpe > -900, "❌ optimizer failed to find a valid portfolio"

    print("5️⃣ Generating briefing...")
    briefing = generate_briefing(weights, sharpe, tickers, sentiment, ub_local, use_quantum)

    print("\n✅ Pipeline completed successfully.\n")
    print(f"Sharpe: {sharpe:.3f}")
    print(dict(zip(tickers, weights.round(3))))
    print("\n--- Briefing ---\n")
    print(briefing)

    return weights, sharpe, briefing

final_weights, final_sharpe, final_briefing = run_full_pipeline(TICKERS, use_quantum=True)

1️⃣ Fetching price data...


/tmp/ipykernel_738/1467081477.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, progress=False)['Close']


2️⃣ Scoring sentiment...
3️⃣ Computing guardrail...
4️⃣ Running optimizer...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


5️⃣ Generating briefing...

✅ Pipeline completed successfully.

Sharpe: 1.382
{'AAPL': np.float64(0.418), 'MSFT': np.float64(0.257), 'GOOGL': np.float64(0.325), 'TSLA': np.float64(0.0), 'AMZN': np.float64(0.0)}

--- Briefing ---

In this portfolio report, we've optimized the asset allocation to maximize returns while minimizing risk, employing Harris Hawks Swarm Optimization—a sophisticated algorithm known for its efficiency in complex problem-solving. To ensure the strategy stays within a sentiment-based guardrail, we've integrated a unique approach leveraging the unpredictable yet deterministic nature of quantum mechanics through Qiskit, a powerful quantum computing framework. This combination allows us to navigate the volatile market landscape with precision and adaptability.

Sharpe Ratio: 1.382

- AAPL: 41.8% allocated | sentiment +0.09 | cap 38.2%
- MSFT: 25.7% allocated | sentiment -0.08 | cap 32.2%
- GOOGL: 32.5% allocated | sentiment -0.04 | cap 33.5%
- TSLA: 0.0% allocated | 

In [36]:
def generate_briefing(weights, sharpe, tickers, sentiment_scores, ub_array, used_quantum=True):
    # Build the exact, guaranteed-correct data table yourself — no LLM involved here
    table = "\n".join(
        f"- {t}: {w*100:.1f}% allocated | sentiment {s:+.2f} | cap {u*100:.1f}%"
        for t, w, s, u in zip(tickers, weights, sentiment_scores, ub_array)
    )

    prompt = f"""Write a 3-sentence plain-English intro paragraph (no numbers, no lists) for a
portfolio report. Mention that allocation was optimized using Harris Hawks swarm optimization with
a sentiment-based guardrail, and randomness source was {"quantum (Qiskit)" if used_quantum else "classical"}.
Keep it general — do not mention any specific tickers, percentages, or numbers."""

    messages = [{"role": "user", "content": prompt}]
    output = llm(messages, max_new_tokens=100, do_sample=False)
    intro = output[0]['generated_text'][-1]['content']

    # Stitch the LLM's intro together with your own verified numbers — model never touches the data
    full_briefing = f"""{intro}

Sharpe Ratio: {sharpe:.3f}

{table}
"""
    return full_briefing

# using best_weights / best_sharpe from cell 20 — these already exist at this point.
# (final_weights/final_sharpe don't exist yet — those only get created later,
# by run_full_pipeline in cell 30)
briefing = generate_briefing(best_weights, best_sharpe, TICKERS, S, UB, used_quantum=True)
print(briefing)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In this portfolio report, we've optimized the asset allocation to maximize returns while minimizing risk, employing Harris Hawks Swarm Optimization—a sophisticated algorithm known for its efficiency in complex problem-solving. To ensure the strategy stays within a sentiment-based guardrail, we've integrated a unique approach leveraging the unpredictable yet deterministic nature of quantum mechanics through Qiskit, a powerful quantum computing framework. This combination allows us to navigate the volatile market landscape with precision and adaptability.

Sharpe Ratio: 1.383

- AAPL: 42.2% allocated | sentiment +0.09 | cap 38.2%
- MSFT: 26.2% allocated | sentiment -0.08 | cap 32.2%
- GOOGL: 31.6% allocated | sentiment -0.04 | cap 33.5%
- TSLA: 0.0% allocated | sentiment -0.19 | cap 28.3%
- AMZN: 0.0% allocated | sentiment +0.16 | cap 40.6%

